In [ ]:
pip install networkit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from scipy.spatial.distance import euclidean
import networkx as nx
from collections import Counter
import scipy.stats as stats
from scipy import io
import pandas as pd
from scipy.io import loadmat
#import networkit as nk
#from GraphRicciCurvature.OllivierRicci import OllivierRicci
import glob
from tqdm import tqdm
import networkx as nx
import numpy as np
import networkit as nk
import warnings
import seaborn as sns
import math

# 1. Generate Networks

## 1.0 load from mat

In [ ]:
from scipy.io import loadmat
from scipy.sparse import csr_matrix

def load_graph_from_mat(cur_file=None):
    # Load the MAT file
    mat = loadmat(cur_file)
    adjacency_matrix = mat['adjSim']
    sparse_matrix = csr_matrix(adjacency_matrix)
    G = nx.from_numpy_array(sparse_matrix)
    if not nx.is_connected(G):
            warnings.warn('The original_graph is not connected. \nLocatoin: ' + cur_file)
            subg = max(nx.connected_components(G), key=len)
            G = G.subgraph(subg)
            G = nx.convert_node_labels_to_integers(G)
    if len(G.nodes()) <= 1000 or len(G.edges()) <= 1000:
            warnings.warn('A original_graph is ignored because it has only a few nodes. \nLocatoin: ' + cur_file)
            return None
    return G
# Load the MAT file
G_list = []
directory = '/home/zhangpeiyu/llm/interpretability_llm/compass/result_weightedGraph_15_AA'
files = os.listdir(directory)
for file in tqdm(files):
    if file.endswith('.mat'):
        G = load_graph_from_mat(directory+'/'+file)
        G_list.append(G)

## 1.1. different kX curve in one pic

In [ ]:
from pickle import LONG
def load_graph_fromcsv(cur_file=None, directed=False):
    # G = nx.read_edgelist(cur_file, delimiter=',', comments='Source', data=[('Weight', float),('Length', float)])  # pay attention to this, 'Length' is true weight, 'Weight' is the diameter
    G = nx.read_edgelist(cur_file, delimiter=';', comments='Source', data=[('Weight', float),('Length', float),('Length in nm', str)])
    for edge in G.edges(data=True):
        edge[2]['weight']=edge[2].pop('Weight')/edge[2].pop('Length')
        # edge[2].pop('Length')
    if len(G.edges()) <= 20:
            warnings.warn('A original_graph is ignored because it has only a few edges. \nLocatoin: ' + cur_file)
            return None
    if not nx.is_connected(G):
            warnings.warn('The original_graph is not connsected. \nLocatoin: ' + cur_file)
            subg = max(nx.connected_components(G), key=len)
            G = G.subgraph(subg)
            G = nx.convert_node_labels_to_integers(G)
    return G

def sort_key(filename):
    x = ''.join(filter(str.isdigit, filename))
    if not x:
      return 0
    num = int(x)
    return num

G_list = []
name = []

root = '/content/drive/MyDrive/Compass_Chung_Man/Edge list and length in nm/'
file_names = os.listdir(root)
print(file_names)
file_names = sorted(file_names, key=sort_key)
for file in file_names:
    if file[-4:] != '.csv' or file == 'Compiled_Data.csv':
        continue
    cur_file = root + file
    G = load_graph_fromcsv(cur_file, directed=False)
    if G is not None:
        G_list.append(G)
        name.append(file.split(' ')[1])
print(name)

## 1.2.  20k 30k 60k 80k(A B C D curve in one pic)

In [ ]:
from pickle import LONG
def load_graph_fromcsv(cur_file=None, directed=False):
    # if cur_file.split('/')[-2] == 'D':
    #   G = nx.read_edgelist(cur_file, delimiter=';', comments='Source', data=[('Weight', float),('Length', float),('Length in nm', str)])  # pay attention to this, 'Length' is true weight, 'Weight' is the diameter
    # else:
    G = nx.read_edgelist(cur_file, delimiter=',', comments='Source', data=[('Weight', float),('Length', float)])  # pay attention to this, 'Length' is true weight, 'Weight' is the diameter
    for edge in G.edges(data=True):
        edge[2]['weight']=edge[2].pop('Weight')/edge[2].pop('Length')

    if len(G.edges()) <= 20:
            warnings.warn('A original_graph is ignored because it has only a few edges. \nLocatoin: ' + cur_file)
            return None
    if not nx.is_connected(G):
            warnings.warn('The original_graph is not connected. \nLocatoin: ' + cur_file)
            subg = max(nx.connected_components(G), key=len)
            G = G.subgraph(subg)
            # G = nx.convert_node_labels_to_integers(G)
    return G

G_list = []
name = []
# root = '/home/zhangpeiyu/llm/interpretability_llm/Edge list and length in nm'
root = '/content/drive/MyDrive/Compass_Chung_Man/compare/'
dirs = ['A','B','C','D']
for dir in dirs:
  file_names = os.listdir(root+dir)
  for file in file_names:
    if '10kX' in file:
      if file[-4:] != '.csv' or file == 'Compiled_Data.csv':
          continue
      cur_file = root + dir + '/' + file
      print(cur_file)
      G = load_graph_fromcsv(cur_file, directed=False)
      if G is not None:
          G_list.append(G)
          name.append(dir)
print(name)


In [ ]:
max_dim = []
min_dim = []
dimension = []
holder_exp = []
widths = []
avg_frac = []
avg_closeness = []
avg_degree = []
avg_clustering = []

## 1.3 Sample subgraph

In [ ]:
import random

def random_connected_subgraph(G, n):
    # random select a node
    node = random.choice(list(G.nodes))
    subgraph = []
    subgraph.append(node)
    while len(subgraph) < n:
        # random select a neighbour
        neighbors = list(nx.neighbors(G, node))
        next_node = random.choice(neighbors)
        if next_node not in subgraph:
          subgraph.append(next_node)
        # subgraph.add_node(node)
        node = random.choice(subgraph)
    subgraph = G.subgraph(subgraph)
    return subgraph

G_list = []
name = []
N = len(G.nodes)
for i in range(100,N,100):
  G_list.append([])
  name.append(str(i)+'nodes')
  for j in range(10):
    G_list[-1].append(random_connected_subgraph(G,i))

## 1.4 Rwrite Graph

In [ ]:
import random
name = []
G_list = []
print(len(G.edges()))
def random_rewrite_graph(G,p):
  G_new = G.copy()
  for e in G.edges(data=True):
      # rewire the edges with probability p
      if np.random.uniform(0,1)<=p and G_new.has_edge(e[0], e[1]):
          G_new.remove_edge(e[0], e[1])
      # check if the rewiring edge is existing
          while True:
            edge_new = random.sample(G.nodes(), 2)
            if not G_new.has_edge(edge_new[0], edge_new[1]):
              break
          G_new.add_edge(edge_new[0], edge_new[1], weight=e[2]['weight'])
          G_new.add_edge(edge_new[0], edge_new[1], weight=e[2]['weight'])
  if not nx.is_connected(G_new):
      # warnings.warn('The original_graph is not connected. \nLocatoin: ' + cur_file)
      subg = max(nx.connected_components(G_new), key=len)
      G_new = G_new.subgraph(subg)
      G_new = nx.convert_node_labels_to_integers(G_new)
  return G_new

for p in np.arange(0.2,1.01,0.1):
  G_list.append([])
  name.append('p = '+str(round(p,1)))
  for j in range(10):
    G_list[-1].append(random_rewrite_graph(G,p))
    # print(len(G_list[-1][-1].edges()))

# 2. Node-based Multifractal Analysis

In [ ]:
#4 NFD for weighted directed network
def wnfd_nk(G,Q,weight=True,draw=False,fdigi=0):
## Find radius
    N_list = []
    r_g_all_set = set()
    num_nodes_all = nx.number_of_nodes(G)
    G = nx.convert_node_labels_to_integers(G)
    if weight == True:
        G_nk = nk.nxadapter.nx2nk(G,weightAttr='weight')
    else:
        G_nk = nk.nxadapter.nx2nk(G)
    for node in tqdm(G.nodes(), total=num_nodes_all):
        grow = []
        grow_ori = nk.distance.Dijkstra(G_nk, node, storePaths=False).run().getDistances()
        for s in grow_ori:
            if s>0 and s<99999:
                grow.append(s)
        grow.sort()
        # upf = (1/pow(10,fdigi+1)*5)
        # grow = [round(d+upf,fdigi) for d in grow]
        if fdigi == 0:
          grow = [math.ceil(d) for d in grow]
        else:
          grow = [round(d,fdigi) for d in grow]
#         grow = grow[1:]
        num = Counter(grow)
        r_g_all_set.update(num.keys())
        N_list.append(num)

    r_g_all = np.array(sorted(list(r_g_all_set)))
    Nw_mat = np.ones((len(N_list), len(r_g_all))) # Num_r matrix: column: node, row: radius

    for i, num in enumerate(N_list):
        for j, r in enumerate(r_g_all):
            Nw_mat[i, j] += sum(count for radius, count in num.items() if radius <= r)

## Distortion factor q: get Zq_mat
    diameter = r_g_all[-1]
    # print('diameter:',diameter)

    Zq_list = []

    for q in Q:
        Zq_mat = np.power(Nw_mat / Nw_mat[:, -1, None], q)
        Zq_list.append(np.sum(Zq_mat, axis=0))

## Get tau(slope)
    tau_list = []
    if draw == True:
        plt.figure(figsize=(7,7))
        for idx, q in enumerate(Q):
            r_g_all_np = np.array(r_g_all)
            x = np.log(r_g_all_np/diameter)
            y = np.log(Zq_list[idx])
            q = format (q, '.0f' )
            plt.plot(x,y,'*',label='q='+str(q))
    #         # plt.plot(x,y,'*',label='q='+str(q))
    #         # plt.legend(fontsize=10)
            slope, intercept, _, _, _  = stats.linregress(x, y)
#             plt.plot(x,intercept + slope*x,alpha=0.5)
            tau_list.append(slope)
            plt.xlabel('ln(r/d)')
            plt.ylabel('ln(sum function)')
    else:
        for idx, q in enumerate(Q):
            r_g_all_np = np.array(r_g_all)
            x = np.log(r_g_all_np/diameter)
            # print('x:',x)
            y = np.log(Zq_list[idx])
            slope, intercept, _, _, _  = stats.linregress(x, y)
            tau_list.append(slope)
    # print('tau_list:',tau_list)

    return tau_list

In [ ]:
def nspectrum(tau_list,q_list,k,color):
    al_list = []
    fal_list = []
    for i in range(1,len(q_list)):
        al=(tau_list[i]-tau_list[i-1])/(q_list[i]-q_list[i-1])
        al_list.append(al)
    for j in range(len(q_list)-1):
        fal=q_list[j]*al_list[j]-tau_list[j]
        fal_list.append(fal)
    #plt.figure(figsize=(10,10))
    plt.plot(al_list,fal_list,label=name[k],linewidth=3,color=color)
    #plt.plot(al_list,fal_list,linewidth=5)
    plt.xlabel('Lipschiz-Hölder exponent, 'r'$\alpha$')
    plt.ylabel('Multi-fractal spectrum, 'r'$f(\alpha)$')
    #plt.legend()
    #plt.savefig('/Users/xiongyex/Downloads'+'/Spec_{}.png'.format(k),bbox_inches = 'tight',dpi=600)
    alpha_0 = al_list[np.argmax(fal_list)]
    width = np.max(al_list) - np.min(al_list)
    print('Holder Exponent:', alpha_0)
    print('width:', width)
    holder_exp.append(alpha_0)
    widths.append(width)
    return alpha_0, width

In [ ]:
def ndimension(tau_list,q_list,k,color):
    dim_list = []
    qd_list = []
    for i in range(len(q_list)):
        if q_list[i] != 0:
            dim = tau_list[i]/q_list[i]
            dim_list.append(dim)
            qd_list.append(q_list[i])
    #plt.figure(figsize=(10,10))
    plt.plot(qd_list,dim_list,label=name[k],linewidth=3,color=color)
    #plt.plot(qd_list,dim_list,linewidth=5)
    plt.xlabel('Distorting exponent, 'r'$q$')
    plt.ylabel('Generalized fractal dimension, 'r'$D(q)$')
    #plt.legend()
    print('Dim_max: ', np.max(dim_list))
    print('Dim_min: ', np.min(dim_list))
    print('Dim_max-min: ',np.max(dim_list)-np.min(dim_list))
    max_dim.append(np.max(dim_list))
    min_dim.append(np.min(dim_list))
    dimension.append(np.max(dim_list)-np.min(dim_list))
    return dim_list
    #plt.savefig('/Users/xiongyex/Downloads'+'/Spec_{}.png'.format(k),bbox_inches = 'tight',dpi=600)

In [ ]:
holder_exp = []
widths = []

plt.rcParams.update({'font.size': 20})
plt.figure(figsize=(10, 7))

Q = [q/100 for q in range(-2000,2001,10)]
wei_ntauls_list = []


for i in range(len(G_list)):
    ntau = wnfd_nk(G_list[i],Q,weight=True,draw=False) #Peiyu: weight=None
    wei_ntauls_list.append(ntau)
np.save(root+'div_ntauls_80kX.npy',wei_ntauls_list)

wei_ntauls_list = np.load(root+'div_ntauls_80kX.npy',allow_pickle=True)

# wei_ntauls_list = np.load('/content/drive/MyDrive/Compass_Chung_Man/compare/'+'wei_ntauls_15kX_2.npy',allow_pickle=True)

colors = sns.color_palette("coolwarm", len(wei_ntauls_list))

for i in range(len(wei_ntauls_list)):
    # nspectrum(wei_ntauls_list[i],Q,i,color=colors[i])
    nspectrum(wei_ntauls_list[i][170:231],Q[170:231],i,color=colors[i])
print(holder_exp)
print(widths)
plt.legend(fontsize=25,loc='best')
# plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.legend(frameon=False, prop={'size': 15}, loc='best')
plt.grid(False)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')
plt.savefig(root+'weighted_80kX_nspectrum.svg',bbox_inches = 'tight',dpi=600)

In [ ]:
max_dim = []
min_dim = []
dimension = []

plt.rcParams.update({'font.size': 20})
plt.figure(figsize=(10,7))

# Q = [q/100 for q in range(-2000,2001,10)]
# ntauls_list = []
# for i in range(len(G_list)):
#     ntau = wnfd_nk(G_list[i],Q,weight=False,draw=False)
#     ntauls_list.append(ntau)

# np.save('ntauls_ndimension_D_unweight.npy',ntauls_list)

# colors = sns.color_palette("coolwarm", len(ntauls_list))
Q = [q/100 for q in range(-2000,2001,10)]
wei_ntauls_list = np.load(root+'div_ntauls_80kX.npy',allow_pickle=True)

for i in range(len(wei_ntauls_list)):
    print(name[i])
    ndimension(wei_ntauls_list[i],Q,i,color=colors[i])
print(dimension)
plt.legend(fontsize=25,loc='best')
# plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.legend(frameon=False, prop={'size': 15}, loc='best')
# plt.legend(ncol=10,frameon=False,prop={'size': 15}, loc='lower center', bbox_to_anchor=(0.5, -0.3))
plt.grid(False)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')
plt.savefig(root+'weighted_80kX_ndimension.svg',bbox_inches = 'tight',dpi=600)

# 3. Parameters Distribution

In [ ]:
from tqdm import tqdm

##  3.1 NFD

In [ ]:
# NFD
def node_dimension(G,weight=True):
    node_dimension = {}
    if weight == None:
        G_nk = nk.nxadapter.nx2nk(G)
    else:
        G_nk = nk.nxadapter.nx2nk(G,weightAttr='weight')

    for node in G.nodes():
        grow = []
        r_g = []
        num_g = []
        num_nodes = 0
        grow = nk.distance.Dijkstra(G_nk, int(node), storePaths=False).run().getDistances()
        grow.sort()
        if weight == True:
          grow = [math.ceil(d) for d in grow]
        grow = grow[1:]
        num = Counter(grow)
        for i,j in num.items():
            num_nodes += j
            if i>0:
                #if np.log(num_nodes) < 0.95*np.log(G.number_of_nodes()):
                r_g.append(i)
                num_g.append(num_nodes)
#                 # delete
#                 if np.log(num_nodes) > 0.9*np.log(G.number_of_nodes()):
#                     break
        x = np.log(r_g)
        y = np.log(num_g)

#         if len(r_g) < 1:
#             print("local",node)
        if len(r_g) > 1:
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
            node_dimension[node] = slope
        else:
            node_dimension[node] = 0
    return node_dimension

In [ ]:
#Fractal Dimension
nfd_centrality_list = []
avg_frac = []
for G in G_list:
    nfd_centrality = node_dimension(G,weight=True).values() #Peiyu: weight=None
    nfd_centrality_list.append(list(nfd_centrality))
    avg_frac.append(np.mean(np.array(nfd_centrality_list[-1])))

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(nfd_centrality_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Fractal Dimension')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_nfd_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 3.2 NFD Node_mass


In [ ]:
# NFD with mass
from collections import defaultdict

def node_dimension_mass(G, node_mass, weight=True):
    """
    Calculate the fractal dimension of each node in the original_graph.

    Parameters:
    G (networkx.Graph): The input original_graph.
    node_mass (dict): A dictionary storing the mass of each node.
    weight (bool): Whether to consider edge weights in the calculation.

    Returns:
    dict: A dictionary with nodes as keys and their calculated dimensions as values.
    """
    node_dimension = {}

    for node in G.nodes():
        distances = nx.single_source_dijkstra_path_length(G, node, weight='weight' if weight else None)

        r_dict = defaultdict(float)

        for target_node, distance in distances.items():
            if distance > 0:
                r_dict[distance] += node_mass[target_node]

        r_all = np.array(sorted(r_dict.keys()))

        if len(r_all) <= 1:
            node_dimension[node] = 0
            continue

        Nw_mat = np.zeros(len(r_all))

        for i, r in enumerate(r_all):
            Nw_mat[i] = sum(mass for dist, mass in r_dict.items() if dist <= r)

        x = np.log(r_all)
        y = np.log(Nw_mat)

        slope, _, _, _, _ = stats.linregress(x, y)
        node_dimension[node] = slope

    return node_dimension

In [ ]:
#Fractal Dimension
nfdmass_centrality_list = []
avg_mass_frac = []
for G in G_list:
    nfdmass_centrality = node_dimension(G,weight=True).values() #Peiyu: weight=None
    nfdmass_centrality_list.append(list(nfdmass_centrality))
    avg_mass_frac.append(np.mean(np.array(nfdmass_centrality_list[-1])))

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(nfdmass_centrality_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Fractal Dimension')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_nfdmass_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

#



## 3.3 Closeness centrality

In [ ]:
# Closeness centrality
closeness_centrality_list = []
avg_closeness = []
for G in tqdm(G_list):
    closeness_centrality = nx.closeness_centrality(G,distance='weight').values()
    closeness_centrality_list.append(list(closeness_centrality))
    avg_closeness.append(np.mean(np.array(closeness_centrality_list[-1])))

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(closeness_centrality_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Closeness Centrality')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_closeness_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 3.4 Degree centrality

In [ ]:
# Degree centrality
degree_centrality_list = []
avg_degree = []

for G in tqdm(G_list):
    degree_centrality = dict(G.degree(weight='weight')).values()
    degree_centrality_list.append(list(degree_centrality))
    avg_degree.append(np.mean(np.array(degree_centrality_list[-1])))


In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
# plt.title('Video 1: Reference')
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(degree_centrality_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Degree Centrality')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_degree_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 3.5 Clustering coefficient

In [ ]:
# Clustering coefficient
cluster_coef_list = []

avg_clustering = []

for G in tqdm(G_list):
    cluster_coef = nx.clustering(G, weight='weight').values()
    cluster_coef_list.append(list(cluster_coef))
    avg_clustering.append(np.mean(np.array(cluster_coef_list[-1])))

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
# plt.title('Video 1: Reference')
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(cluster_coef_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Clustering Coefficient')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_clustering_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 3.6 Betweeness

In [ ]:

# Calculate average betweenness centrality
betweenness_list = []
avg_betweenness_list = []
for G_nx in tqdm(G_list):
    G = nk.nxadapter.nx2nk(G_nx, weightAttr='weight')
    betweenness = nk.centrality.Betweenness(G, normalized=True).run()
    betweenness_list.append(betweenness.scores())
    avg_betweenness = sum(betweenness.scores()) / G.numberOfNodes()
    avg_betweenness_list.append(avg_betweenness)

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
# plt.title('Video 1: Reference')
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(betweenness_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Betweenness Centrality')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_betweenness_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 3.7 Graph Ollivier-Ricci

In [ ]:
!pip install GraphRicciCurvature
!pip install scikit-learn

In [ ]:
import networkx as nx
from GraphRicciCurvature.OllivierRicci import OllivierRicci

In [ ]:
avg_orcs = []
orc_list = []
for G in tqdm(G_list):
  avg_orc = []
  orc = OllivierRicci(nx.convert_node_labels_to_integers(G), alpha=0.5, verbose="ERROR")
  # 计算Ollivier-Ricci曲率
  orc.compute_ricci_curvature()
  G_orc = orc.G.copy()
  # 输出每条边的Ollivier-Ricci曲率
  for (u, v, d) in G_orc.edges(data=True):
    avg_orc.append(d['ricciCurvature'])
  orc_list.append(avg_orc)
  avg_orcs.append(np.mean(np.array(avg_orc)))

In [ ]:
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(10,7))
plt.grid(False)
# plt.title('Video 1: Reference')
for i,t in enumerate(name): #Peiyu:modify
    sns.kdeplot(list(orc_list[i]),linewidth=1.8, label=t,color=colors[i])

plt.legend(frameon=False,loc='best',fontsize=15)
#plt.ylim([0,2.1])
plt.xlabel('Graph Ollivier-Ricci')

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

plt.savefig(root+'weighted_Ollivier-Ricci_80kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

# 4. Save Data

##  4.1 save avg to csv

In [ ]:
import pandas as pd

data = {
    'holder_exp': holder_exp,
    'widths': widths,
    'dimension': dimension,
    'avg_frac': avg_frac,
    'avg_closeness': avg_closeness,
    'avg_degree': avg_degree,
    'avg_clustering': avg_clustering,
    'avg_betweenness_list': avg_betweenness_list,
    'avg_orcs': avg_orcs
}

df = pd.DataFrame(data, index=['A', 'B', 'C', 'D'])
df = df.transpose()
df.to_csv(root+'output.csv')

## 4.2 Radar Chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 18})

labels = np.array(['Holder\nExponent', 'Width', 'Dim(max-min)', 'Avg.Degree', 'Ollivier-Ricci\ncurvature'])
angles = np.arange(0, 360, 360.0/len(labels))


angles += angles[:1]
fig = plt.figure(figsize=(8,8))
# 绘制图形
rect = [0.05, 0.05, 0.9, 0.9]
axes = [fig.add_axes(rect, projection='polar', label=label) for label in labels]

ax = axes[0]
ax.set_thetagrids(angles, labels=labels, fontsize=15, fontweight='bold')
for ax in axes[1:]:
    ax.patch.set_visible(False)
    ax.grid(False)
    ax.xaxis.set_visible(False)

epoch=[
    [0.1,0.35,0.6,0.85,1.1,1.35],
    [0.5,1,1.5,2,2.5,3],
    [1.6,1,9.2,2,2.5,2.8,3.1],
    [0.05,0.15,0.25,0.35,0.45,0.55],
    [-0.0,-0.15,-0.3,-0.45,-0.6,-0.75]
]

labs = []
for ep in epoch:
  labs.append([str(item) for item in ep[1:]])

for ax,angle,label,i in zip(axes,angles,labs,epoch):
            ax.set_rgrids(i[1:-1],angle=angle,labels=label[:-1],fontsize=10)
            ax.spines['polar'].set_visible(False)
            ax.set_ylim(i[0],i[5])
ax = axes[0]
angles = np.deg2rad(np.r_[angles,angles[0]])
for i in range(len(name)):
  data = np.array([holder_exp[i],widths[i],dimension[i],avg_degree[i],avg_orcs[i]])
  print(data)
  limits = []
  limits.append((data[1]-0.5)/0.5)
  limits.append((data[2]-1.6)/0.3)
  limits.append((data[3]-0.05)/0.1)
  limits.append((data[4])/(-0.15))
  data[1] = (limits[0])*0.25+0.1
  data[2] = (limits[1])*0.25+0.1
  data[3] = (limits[2])*0.25+0.1
  data[4] = (limits[3])*0.25+0.1
  data = np.r_[data,data[0]]
  ax.plot(angles, data, linewidth=1.5,linestyle='solid',alpha=1,color=colors[i],label=name[i])
  ax.scatter(angles, data, color=colors[i])
  # ax.set_rticks([1, 2, 3, 4, 5])
  # ax.set_thetagrids(np.degrees(angles), labels, fontsize=12)
  ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=15)
plt.savefig(root+'radar_10kX.svg',bbox_inches = 'tight',dpi=600)
plt.show()

## 4.3 load as sparse_matrices

In [ ]:
from scipy.sparse import save_npz

sparse_matrices = {}
for i, G in enumerate(G_list):
    nos = G.nodes()
    nos = [int(nn) for nn in nos]
    nos = sorted(nos)
    nos = [str(nn) for nn in nos]
    sparse_matrix = nx.adjacency_matrix(G,nodelist=nos)
    sparse_matrices[name[i]] = sparse_matrix

np.savez(root+'sparse_matrices.npz', **sparse_matrices)

## 4.4 ERROR BAR

### 4.4.1 load_network


In [ ]:
from pickle import LONG
from scipy.sparse import csr_matrix

root = '/content/drive/MyDrive/Compass_Chung_Man/compare'
cur_file = root + '/' + 'sparse_matrices.npz'
data = np.load(cur_file, allow_pickle=True)
sparse_matrix = data['A']
sparse_matrix = csr_matrix(sparse_matrix.all())
G = nx.from_numpy_array(sparse_matrix)
print(G.nodes())


### 4.4.2 Calculates avgs

In [ ]:
#Fractal Dimension
avg_frac = []
# Closeness centrality
avg_closeness = []
# Degree centrality
avg_degree = []
# Clustering coefficient
avg_clustering = []

for G in G_list:
    nfd_centrality = node_dimension(G,weight=True).values()
    avg_frac.append(np.mean(np.array(list(nfd_centrality))))
    closeness_centrality = nx.closeness_centrality(G,distance='weight').values()
    avg_closeness.append(np.mean(np.array(list(closeness_centrality))))
    degree_centrality = dict(G.degree(weight='weight')).values()
    avg_degree.append(np.mean(np.array(list(degree_centrality))))
    cluster_coef = nx.clustering(G, weight='weight').values()
    avg_clustering.append(np.mean(np.array(list(cluster_coef))))
print(avg_frac)
print(avg_closeness)
print(avg_degree)
print(avg_clustering)


In [ ]:
def get_mean_std(nums):
  mean = sum(nums) / len(nums)
  variance = sum((num - mean) ** 2 for num in nums) / (len(nums) - 1)
  return mean, variance

avgs = [avg_frac, avg_closeness, avg_degree, avg_clustering]
y = []
yerr = []
for avg in avgs:
  mean, variance = get_mean_std(avg)
  y.append(mean)
  yerr.append(variance)


In [ ]:
import matplotlib.pyplot as plt

# Simulate some data
x = ['nfd', 'closeness', 'degree', 'clustering'] #properties

# Plot data with error bars
plt.errorbar(x, y, yerr=yerr, fmt='o', capsize=5, capthick=2, ecolor='red', markerfacecolor='blue', label='Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Error Bar Plot')
plt.legend()
plt.show()